# Micrograd — From Scratch

A ground-up implementation of Andrej Karpathy's **micrograd**: a tiny scalar-valued autograd engine with a PyTorch-like API, built entirely in pure Python.

**What this notebook covers:**
1. Numerical differentiation basics
2. The `Value` object — scalar autograd engine
3. Computation-graph visualisation with Graphviz
4. Backpropagation through a single neuron (tanh, manual breakdown)
5. Debugging gradient accumulation bugs
6. Cross-validation against PyTorch
7. Building a full Multi-Layer Perceptron (MLP) from scratch
8. Training loop with MSE loss and gradient descent

## 1. Imports

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Numerical Differentiation

Before building the autograd engine, we ground ourselves in the definition of a derivative.

Given a function `f`, the derivative at point `x` is:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

We use a tiny `h` to approximate this numerically.

In [ ]:
def f(x):
    """A simple scalar function: f(x) = 3x² - 4x + 5"""
    return 3*x**2 - 4*x + 5

# Plot f(x) over [-5, 5]
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.figure(figsize=(7, 3))
plt.plot(xs, ys, color='steelblue', linewidth=2)
plt.title("f(x) = 3x² − 4x + 5")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Numerical derivative of f at x = 2/3
# f'(x) = 6x - 4  →  f'(2/3) = 0.0  (minimum of the parabola)
h = 1e-8
x = 2/3
numerical_derivative = (f(x + h) - f(x)) / h
print(f"f'(2/3) ≈ {numerical_derivative:.6f}  (expected: 0.0)")

### Numerical derivatives of a multi-variable expression

For `d = a·b + c`, the partial derivatives are:
- `∂d/∂a = b`
- `∂d/∂b = a`
- `∂d/∂c = 1`

In [ ]:
h = 1e-4
a, b, c = 2.0, -3.0, 10.0
d = a*b + c
print(f"d = {d}")

# ∂d/∂a
d2 = (a + h)*b + c
print(f"\n∂d/∂a ≈ {(d2 - d)/h:.4f}  (expected: {b})")

# ∂d/∂b
d2 = a*(b + h) + c
print(f"∂d/∂b ≈ {(d2 - d)/h:.4f}  (expected: {a})")

# ∂d/∂c
d2 = a*b + (c + h)
print(f"∂d/∂c ≈ {(d2 - d)/h:.4f}  (expected: 1.0)")

## 3. The `Value` Object — Scalar Autograd Engine

The `Value` class wraps a scalar number and records:
- Its **data** (the scalar value)
- Its **gradient** w.r.t. the final loss (`grad`)
- The **operation** that produced it (`_op`)
- Its **children** in the computation graph (`_prev`)
- A `_backward` closure that propagates gradients to its children

Supported operations: `+`, `*`, `**`, `-`, `/`, unary `-`, `tanh`, `exp`.

In [ ]:
class Value:
    """Scalar value with autograd support."""

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    # ── Arithmetic ──────────────────────────────────────────────────────────

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):          # other + self
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):          # other * self
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Exponent must be int or float"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):       # self / other  →  self * other^-1
        return self * other**-1

    def __neg__(self):                  # -self
        return self * -1

    def __sub__(self, other):           # self - other
        return self + (-other)

    # ── Activations ─────────────────────────────────────────────────────────

    def tanh(self):
        """Hyperbolic tangent activation (fused kernel)."""
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        """e^x activation."""
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    # ── Backpropagation ─────────────────────────────────────────────────────

    def backward(self):
        """Compute gradients via reverse-mode autodiff (backprop)."""
        # Build topological order of the computation graph
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Seed gradient of the output node, then back-propagate
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

### Quick sanity check — arithmetic operations

In [ ]:
a = Value(2.0)
b = Value(4.0)
print("a + b =", a + b)
print("a * b =", a * b)
print("a - b =", a - b)
print("a / b =", a / b)
print("a ** 3 =", a ** 3)

## 4. Computation Graph Visualisation

`draw_dot` renders the full computation graph using Graphviz. Each node shows its label, data, and current gradient.

In [ ]:
from graphviz import Digraph

def trace(root):
    """Collect all nodes and edges in the computation graph."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    """Return a Graphviz Digraph of the computation graph rooted at `root`."""
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(
            name=uid,
            label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad),
            shape='record'
        )
        if n._op:
            dot.node(name=uid + n._op, label=n._op, shape='circle')
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

## 5. Backpropagation Through a Single Neuron

A single neuron computes:

$$o = \tanh\!(x_1 w_1 + x_2 w_2 + b)$$

We build this expression using `Value` objects, call `.backward()`, and verify that gradients flow correctly through the graph.

### 5a. Using the fused `tanh` kernel

In [ ]:
# Inputs
x1 = Value(2.0,  label='x1')
x2 = Value(0.0,  label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0,  label='w2')

# Bias (chosen so n ≈ 0 → tanh(n) ≈ 0.7)
b  = Value(6.8813735870195432, label='b')

# Forward pass
x1w1 = x1 * w1;           x1w1.label = 'x1*w1'
x2w2 = x2 * w2;           x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2;  x1w1x2w2.label = 'x1w1+x2w2'
n = x1w1x2w2 + b;         n.label = 'n'
o = n.tanh();              o.label = 'o'

# Backward pass
o.backward()

print(f"o        = {o.data:.6f}")
print(f"∂o/∂x1   = {x1.grad:.6f}  (expected: w1 * (1-tanh²(n)))")
print(f"∂o/∂w1   = {w1.grad:.6f}  (expected: x1 * (1-tanh²(n)))")

draw_dot(o)

### 5b. Decomposing `tanh` manually into primitive operations

Instead of the fused `tanh`, we can expand:

$$\tanh(n) = \frac{e^{2n}-1}{e^{2n}+1}$$

using only `exp`, `+`, `-`, `/`. The gradients must be identical.

In [ ]:
# Same inputs / weights / bias as above
x1 = Value(2.0,  label='x1')
x2 = Value(0.0,  label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0,  label='w2')
b  = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1;           x1w1.label = 'x1*w1'
x2w2 = x2 * w2;           x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2;  x1w1x2w2.label = 'x1w1+x2w2'
n = x1w1x2w2 + b;         n.label = 'n'

# Manual tanh:  o = (e^2n - 1) / (e^2n + 1)
e = (2 * n).exp()
o = (e - 1) / (e + 1)
o.label = 'o'

o.backward()

print(f"o      = {o.data:.6f}")
print(f"∂o/∂x1 = {x1.grad:.6f}")
print(f"∂o/∂w1 = {w1.grad:.6f}")

draw_dot(o)

## 6. Debugging — Gradient Accumulation

A subtle bug arises when the **same node appears more than once** in an expression. The backward pass must *accumulate* (+=) gradients, not overwrite them (=).

Two canonical test cases are shown below.

### Case 1 — Node used twice in addition (`b = a + a`)

In [ ]:
a = Value(3.0, label='a')
b = a + a;  b.label = 'b'
b.backward()

# ∂b/∂a = 2  (a appears twice, so gradients must accumulate)
print(f"a.grad = {a.grad}  (expected: 2.0)")
draw_dot(b)

### Case 2 — Node reused across branches (`f = d * e` where `d = a*b`, `e = a+b`)

In [ ]:
a = Value(-2.0, label='a')
b = Value( 3.0, label='b')

d = a * b;  d.label = 'd'          # d = ab
e = a + b;  e.label = 'e'          # e = a + b
f = d * e;  f.label = 'f'          # f = ab(a+b)

f.backward()

# Analytical: ∂f/∂a = b(a+b) + ab = b(a+b+a) = 3(-2+3-2) = 3*(-1) ??? let's just check
print(f"a.grad = {a.grad}")
print(f"b.grad = {b.grad}")
draw_dot(f)

## 7. Cross-Validation Against PyTorch

We replicate the same single-neuron forward/backward pass using PyTorch to confirm our `Value` engine produces numerically identical gradients.

In [ ]:
import torch

x1 = torch.tensor([2.0]).double();                 x1.requires_grad = True
x2 = torch.tensor([0.0]).double();                 x2.requires_grad = True
w1 = torch.tensor([-3.0]).double();                w1.requires_grad = True
w2 = torch.tensor([1.0]).double();                 w2.requires_grad = True
b  = torch.tensor([6.8813735870195432]).double();   b.requires_grad = True

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

o.backward()

print(f"o  = {o.data.item():.6f}")
print(f"x2 = {x2.grad.item():.6f}")
print(f"w2 = {w2.grad.item():.6f}")
print(f"x1 = {x1.grad.item():.6f}")
print(f"w1 = {w1.grad.item():.6f}")

## 8. Building a Neural Network from Scratch

Using `Value` as the primitive, we compose a full **Multi-Layer Perceptron (MLP)**:

```
Neuron  →  Layer  →  MLP
```

### 8a. `Neuron`

In [ ]:
class Neuron:
    """A single neuron: o = tanh(w·x + b)"""

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"Neuron(nin={len(self.w)})"

# Smoke test
x = [2.0, 3.0]
neuron = Neuron(2)
print(neuron(x))

### 8b. `Layer`

In [ ]:
class Layer:
    """A layer of `nout` neurons, each taking `nin` inputs."""

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"

# Smoke test
layer = Layer(2, 3)
print(layer([2.0, 3.0]))

### 8c. `MLP` — Multi-Layer Perceptron

In [ ]:
class MLP:
    """
    Multi-Layer Perceptron.

    Parameters
    ----------
    nin   : number of input features
    nouts : list of neuron counts per layer, e.g. [4, 4, 1]
    """

    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


# Architecture: 3 inputs → [4] → [4] → [1] output
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
print(n)
print(f"\nForward pass output: {n(x)}")
print(f"Total parameters   : {len(n.parameters())}")

## 9. Training the MLP — Binary Classification

We train the MLP on a small dataset of 4 examples with targets `+1` or `-1`
using **Mean Squared Error (MSE)** loss and vanilla gradient descent.

**Training loop:**
1. **Forward pass** — compute predictions and loss
2. **Zero gradients** — reset `.grad` to 0 before backprop
3. **Backward pass** — call `loss.backward()` to fill all `.grad` fields
4. **Parameter update** — nudge each parameter opposite its gradient

In [ ]:
# Dataset
xs = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]   # desired targets

# Fresh network
random.seed(42)
n = MLP(3, [4, 4, 1])

learning_rate = 0.05
losses = []

for k in range(20):
    # 1. Forward pass
    ypred = [n(x) for x in xs]
    loss  = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)], Value(0.0))

    # 2. Zero gradients
    for p in n.parameters():
        p.grad = 0.0

    # 3. Backward pass
    loss.backward()

    # 4. Update parameters
    for p in n.parameters():
        p.data -= learning_rate * p.grad

    losses.append(loss.data)
    print(f"Step {k:2d}  |  loss = {loss.data:.6f}")

### Loss curve

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(losses, marker='o', markersize=4, linewidth=2, color='steelblue')
plt.title("Training Loss (MSE)")
plt.xlabel("Step"); plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Final predictions

In [ ]:
print(f"{'Input':<25} {'Target':>8} {'Predicted':>12}")
print("-" * 47)
for x, ygt, yout in zip(xs, ys, ypred):
    print(f"{str(x):<25} {ygt:>8.1f} {yout.data:>12.6f}")

## 10. Summary

| Component | What it does |
|---|---|
| `Value` | Wraps a scalar; stores data, grad, children, and a `_backward` closure |
| `__add__`, `__mul__`, `__pow__`, … | Build computation graph edges on each operation |
| `backward()` | Topological sort + reverse pass to propagate gradients |
| `Neuron` | `tanh(w·x + b)` with learnable weights and bias |
| `Layer` | Array of neurons with shared input |
| `MLP` | Stack of layers; full forward pass via `__call__` |

**Key insights:**
- Gradients must be *accumulated* (`+=`), not overwritten, when a node appears in multiple branches.
- Zeroing gradients before each backward pass is essential in a training loop.
- Numerical results match PyTorch's autograd engine exactly (for matching architectures).

> *Based on Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) lecture — re-implemented and extended.*